# Stage 6.3 -- T-side A/B Candidate Promotion

Load the candidate baseline promotion package generated from Stage 6.2 outputs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE = Path('../data/gold/modeling/t_side_ab_candidate')

def load_table(name):
    return pd.read_parquet(BASE / f'{name}.parquet')

selection = load_table('candidate_model_selection')
metrics = load_table('candidate_model_metrics')
confusion = load_table('candidate_model_confusion_matrix')
predictions = load_table('candidate_model_predictions')
errors = load_table('candidate_model_errors')
feature_set = load_table('candidate_model_feature_set')
importance = load_table('candidate_model_feature_importance')
comparison = load_table('candidate_model_comparison_vs_baseline')
decision = load_table('candidate_model_decision')
audit = load_table('candidate_model_audit')

## Candidate Selection

In [ ]:
display(selection)

## Metrics

In [ ]:
metric_cols = ['accuracy', 'balanced_accuracy', 'macro_f1', 'f1_A', 'f1_B', 'recall_A', 'recall_B', 'support_A', 'support_B', 'total_errors', 'B_predicted_as_A']
display(metrics[metric_cols])

## Confusion Matrix

In [ ]:
matrix = confusion.pivot(index='true_label', columns='predicted_label', values='count').fillna(0)
display(matrix)
ax = matrix.plot(kind='bar', figsize=(6, 4), rot=0)
ax.set_title('Candidate confusion matrix')
ax.set_xlabel('True label')
ax.set_ylabel('Rounds')
plt.tight_layout()

## Errors

In [ ]:
display(errors[['round_feature_id', 'opponent', 'round_num', 'true_label', 'predicted_label', 'prediction_confidence', 'error_type', 'suggested_review_priority']].head(20))
display(errors['error_type'].value_counts(dropna=False).rename_axis('error_type').reset_index(name='rounds'))

## Feature Set

In [ ]:
summary = feature_set[feature_set['feature_name'].eq('__feature_set_summary__')]
display(summary[['horizon_seconds', 'feature_set_name', 'total_selected_features', 'numeric_features', 'categorical_features', 'notes']])
display(feature_set[~feature_set['feature_name'].eq('__feature_set_summary__')][['feature_name', 'feature_group', 'window_start', 'window_end', 'window_type']].head(30))

## Top Features

In [ ]:
top_features = importance.sort_values('importance_rank').head(15)
display(top_features[['feature_name', 'feature_group', 'importance_value', 'importance_rank', 'direction']])
ax = top_features.sort_values('importance_value').plot.barh(x='feature_name', y='importance_value', figsize=(8, 6), legend=False)
ax.set_title('Top candidate feature importance')
ax.set_xlabel('Importance value')
ax.set_ylabel('Feature')
plt.tight_layout()

## Comparison Against Stage 6 Baseline

In [ ]:
display(comparison[['refined_macro_f1', 'baseline_macro_f1', 'delta_macro_f1', 'refined_recall_B', 'baseline_recall_B', 'delta_recall_B', 'delta_B_predicted_as_A', 'comparison_status']])

## Final Decision

In [ ]:
display(decision)
display(audit)

Next: complete manual review or prepare final project report